# Complete Benchmark Workflow Example

This notebook demonstrates the complete workflow:
1. Define a simple benchmark
2. Run the benchmark
3. Analyze and visualize results

Results are saved to `tests/<benchmark_name>/results/` by default.

## Setup

Import required modules and configure environment.

In [ ]:
import sys
import asyncio
from pathlib import Path

# Ensure project root is in path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Import benchmark library
from benchmark.benchmark import Benchmark
from benchmark.models import Task, TaskResult, Grade
from benchmark.evaluation.graders import exact_match, substring_match

# Import visualizer
from visualizer import ResultsAggregator, ResultsFormatter, GRAPH_AVAILABLE

if GRAPH_AVAILABLE:
    from visualizer import GraphGenerator
    print("✓ Visualization dependencies available")
else:
    print("⚠ Visualization dependencies not installed")
    print("  Install with: pip install -e '.[viz]'")

## Part 1: Define a Custom Benchmark

Create a simple benchmark that tests basic Q&A capabilities.

For this example, we'll use a benchmark that should already exist in `tests/example_benchmark/`.

In [ ]:
class ExampleBenchmark(Benchmark):
    """Example benchmark with simple exact-match grading."""
    
    async def grade(self, result: TaskResult, task: Task) -> Grade:
        """
        Grade task result using exact and substring matching.
        
        Args:
            result: Task execution result
            task: Original task definition
            
        Returns:
            Grade with score and reasoning
        """
        response = result.response.strip().lower()
        expected = str(task.ground_truth).lower()
        
        # Exact match gets full credit
        if exact_match(response, expected, case_sensitive=False):
            return Grade(
                score=1.0,
                passed=True,
                reasoning="Exact match",
                grader_name="exact_match"
            )
        
        # Substring match gets partial credit
        if substring_match(response, expected, case_sensitive=False):
            return Grade(
                score=0.5,
                passed=False,
                reasoning="Partial match (substring found)",
                grader_name="substring_match"
            )
        
        # No match
        return Grade(
            score=0.0,
            passed=False,
            reasoning="No match",
            grader_name="exact_match"
        )

print("✓ Benchmark class defined")

## Part 2: Run the Benchmark

Execute the benchmark against configured models. Results will be saved to `tests/example_benchmark/results/`.

In [ ]:
async def run_example_benchmark():
    """Run the example benchmark and return results."""
    print("Initializing benchmark...")
    benchmark = ExampleBenchmark()
    
    print(f"Benchmark: {benchmark.benchmark_name}")
    print(f"Tasks loaded: {len(benchmark.tasks)}")
    print(f"Models configured: {len(benchmark.config.models)}")
    
    if not benchmark.tasks:
        print("\n⚠ No tasks found!")
        print("  Make sure tests/example_benchmark/questions.jsonl exists")
        return None
    
    if not benchmark.config.models:
        print("\n⚠ No models configured!")
        print("  Add models to tests/example_benchmark/config.yaml")
        return None
    
    print("\nRunning benchmark...\n")
    print("=" * 60)
    
    # Run the benchmark
    output = await benchmark.run()
    
    print("\n" + "=" * 60)
    print("\n✓ Benchmark completed!")
    print(f"  Total tasks: {output.summary['total_tasks']}")
    print(f"  Mean score: {output.summary['mean_score']:.3f}")
    print(f"  Pass rate: {output.summary['pass_rate']:.1%}")
    print(f"  Execution time: {output.execution_time:.2f}s")
    
    # Results are automatically saved to tests/example_benchmark/results/
    print(f"\n  Results saved to: tests/{benchmark.benchmark_name}/results/")
    
    return output

# Run the benchmark
benchmark_output = await run_example_benchmark()

## Part 3: Load and Aggregate Results

Load the saved results and perform aggregation.

In [ ]:
# Initialize aggregator
aggregator = ResultsAggregator()

# Find result files in the default location
results_dir = Path("tests/example_benchmark/results")

if results_dir.exists():
    result_files = sorted(list(results_dir.glob("*.json")))
    print(f"Found {len(result_files)} result file(s) in {results_dir}")
    
    if result_files:
        # Show the files
        for f in result_files:
            print(f"  - {f.name}")
        
        # Load and aggregate
        results = aggregator.aggregate_files(result_files)
        print(f"\n✓ Loaded {len(results)} total results")
    else:
        print("\n⚠ No result files found. Run the benchmark first.")
        results = []
else:
    print(f"⚠ Results directory not found: {results_dir}")
    print("  Run the benchmark first to generate results.")
    results = []

## Part 4: Calculate Summary Statistics

In [ ]:
if results:
    # Overall statistics
    stats = aggregator.calculate_summary_stats(results)
    
    print("Overall Statistics:")
    print("=" * 40)
    print(f"  Count:      {stats['count']}")
    print(f"  Mean:       {stats['mean']:.3f}")
    print(f"  Median:     {stats['median']:.3f}")
    print(f"  Std Dev:    {stats['stddev']:.3f}")
    print(f"  Min:        {stats['min']:.3f}")
    print(f"  Max:        {stats['max']:.3f}")
    
    if 'percentiles' in stats and stats['percentiles']:
        print("\n  Percentiles:")
        for p, val in sorted(stats['percentiles'].items()):
            print(f"    {p:>4}: {val:.3f}")
else:
    print("⚠ No results to analyze. Run the benchmark first.")

## Part 5: Per-Model Analysis

In [ ]:
if results:
    # Aggregate by model
    by_model = aggregator.aggregate_model_results(results)
    
    print("Per-Model Statistics:")
    print("=" * 60)
    
    for model_name, metrics in sorted(by_model.items()):
        print(f"\n{model_name}:")
        print(f"  Tasks:              {metrics['count']}")
        print(f"  Mean Score:         {metrics['mean']:.3f}")
        print(f"  Median Score:       {metrics['median']:.3f}")
        print(f"  Pass Rate:          {metrics['pass_rate']:.1%}")
        print(f"  Passed/Failed:      {metrics['total_passed']}/{metrics['total_failed']}")
        print(f"  Avg Execution Time: {metrics['avg_execution_time']:.2f}s")
        print(f"  Total Time:         {metrics['total_execution_time']:.2f}s")
        
        if 'token_usage' in metrics and metrics['token_usage']:
            total_tokens = metrics['token_usage'].get('total_tokens', 0)
            print(f"  Total Tokens:       {total_tokens:,}")
            
            if 'avg_tokens_per_task' in metrics:
                avg_tokens = metrics['avg_tokens_per_task'].get('total_tokens', 0)
                print(f"  Avg Tokens/Task:    {avg_tokens:,.0f}")

## Part 6: Format Results

In [ ]:
if results:
    formatter = ResultsFormatter()
    
    # Markdown table
    print("\nMarkdown Table:")
    print("=" * 60)
    markdown_table = formatter.to_markdown_table(by_model)
    print(markdown_table)
    
    # Summary
    print("\n" + formatter.format_summary({
        'total_tasks': sum(m['count'] for m in by_model.values()),
        'mean_score': stats['mean'],
        'median_score': stats['median'],
        'pass_rate': sum(m['total_passed'] for m in by_model.values()) / sum(m['count'] for m in by_model.values()),
        'by_model': {name: {'mean_score': m['mean'], 'pass_rate': m['pass_rate']} for name, m in by_model.items()}
    }))

## Part 7: Export Results

In [ ]:
if results:
    # Create analysis directory in the benchmark folder
    analysis_dir = Path("tests/example_benchmark/analysis")
    analysis_dir.mkdir(exist_ok=True)
    
    # Export to CSV
    csv_path = analysis_dir / "model_results.csv"
    formatter.to_csv(by_model, csv_path)
    print(f"✓ Exported to {csv_path}")
    
    # Export to JSON
    json_path = analysis_dir / "model_results.json"
    formatter.to_json(by_model, json_path)
    print(f"✓ Exported to {json_path}")

## Part 8: Visualize Results

Generate plots and charts (requires viz dependencies).

In [ ]:
if results and GRAPH_AVAILABLE:
    grapher = GraphGenerator()
    plots_dir = Path("tests/example_benchmark/analysis/plots")
    plots_dir.mkdir(parents=True, exist_ok=True)
    
    print("Generating visualizations...\n")
    
    # 1. Model comparison - Mean score
    print("1. Model comparison (mean score)...")
    grapher.plot_model_comparison(
        by_model,
        metric="mean",
        output_path=plots_dir / "model_comparison_mean.png"
    )
    print(f"   ✓ Saved to {plots_dir / 'model_comparison_mean.png'}")
    
    # 2. Model comparison - Pass rate
    print("\n2. Model comparison (pass rate)...")
    grapher.plot_model_comparison(
        by_model,
        metric="pass_rate",
        output_path=plots_dir / "model_comparison_pass_rate.png"
    )
    print(f"   ✓ Saved to {plots_dir / 'model_comparison_pass_rate.png'}")
    
    # 3. Score distribution
    print("\n3. Score distribution...")
    grouped = aggregator.group_by_model(results)
    grapher.plot_score_distribution(
        grouped,
        output_path=plots_dir / "score_distribution.png"
    )
    print(f"   ✓ Saved to {plots_dir / 'score_distribution.png'}")
    
    # 4. Timeline
    print("\n4. Score timeline...")
    grapher.plot_timeline(
        results,
        output_path=plots_dir / "timeline.png"
    )
    print(f"   ✓ Saved to {plots_dir / 'timeline.png'}")
    
    # 5. Execution time comparison
    print("\n5. Execution time comparison...")
    grapher.plot_execution_time_comparison(
        by_model,
        output_path=plots_dir / "execution_times.png"
    )
    print(f"   ✓ Saved to {plots_dir / 'execution_times.png'}")
    
    print(f"\n✓ All plots saved to {plots_dir}/")
    
elif results:
    print("⚠ Visualization dependencies not installed")
    print("  Install with: pip install -e '.[viz]'")
else:
    print("⚠ No results to visualize. Run the benchmark first.")

## Part 9: Per-Task Analysis

Analyze performance on individual tasks to identify difficult questions.

In [ ]:
if results:
    # Aggregate by task
    by_task = aggregator.aggregate_task_results(results)
    
    print("Per-Task Analysis (Hardest Tasks):")
    print("=" * 60)
    
    # Sort by mean score (ascending = hardest first)
    sorted_tasks = sorted(by_task.items(), key=lambda x: x[1]['mean'])
    
    # Show top 10 hardest
    for i, (task_id, metrics) in enumerate(sorted_tasks[:10], 1):
        print(f"\n{i}. Task: {task_id}")
        print(f"   Mean Score:     {metrics['mean']:.3f}")
        print(f"   Median Score:   {metrics['median']:.3f}")
        print(f"   Pass Rate:      {metrics['pass_rate']:.1%}")
        print(f"   Models Tested:  {metrics['models_tested']}")

# Benchmark Results Analysis

This notebook demonstrates how to use the visualizer module to analyze benchmark results.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from visualizer import ResultsAggregator, ResultsFormatter, GRAPH_AVAILABLE

if GRAPH_AVAILABLE:
    from visualizer import GraphGenerator
else:
    print("Visualization dependencies not installed. Install with: pip install -e '.[viz]'")

## 1. Load and Aggregate Results

In [ ]:
# Initialize aggregator
aggregator = ResultsAggregator()

# Find all result files
results_dir = Path("results")
result_files = list(results_dir.glob("*.json"))
print(f"Found {len(result_files)} result files")

# Load and aggregate
if result_files:
    results = aggregator.aggregate_files(result_files)
    print(f"Loaded {len(results)} total results")
else:
    print("No result files found. Run a benchmark first with: python main.py --benchmark <name>")
    results = []

## 2. Calculate Summary Statistics

In [ ]:
if results:
    # Overall statistics
    stats = aggregator.calculate_summary_stats(results)
    print("Overall Statistics:")
    print(f"  Count: {stats['count']}")
    print(f"  Mean: {stats['mean']:.3f}")
    print(f"  Median: {stats['median']:.3f}")
    print(f"  Std Dev: {stats['stddev']:.3f}")
    print(f"  Min: {stats['min']:.3f}")
    print(f"  Max: {stats['max']:.3f}")
    
    if 'percentiles' in stats:
        print("  Percentiles:")
        for p, val in stats['percentiles'].items():
            print(f"    {p}: {val:.3f}")

## 3. Per-Model Analysis

In [ ]:
if results:
    # Aggregate by model
    by_model = aggregator.aggregate_model_results(results)
    
    print("\nPer-Model Statistics:\n")
    for model_name, metrics in sorted(by_model.items()):
        print(f"{model_name}:")
        print(f"  Tasks: {metrics['count']}")
        print(f"  Mean Score: {metrics['mean']:.3f}")
        print(f"  Pass Rate: {metrics['pass_rate']:.1%}")
        print(f"  Avg Execution Time: {metrics['avg_execution_time']:.2f}s")
        if 'token_usage' in metrics:
            print(f"  Total Tokens: {metrics['token_usage'].get('total_tokens', 0):,}")
        print()

## 4. Format as Markdown Table

In [ ]:
if results:
    formatter = ResultsFormatter()
    markdown_table = formatter.to_markdown_table(by_model)
    print(markdown_table)

## 5. Export Results

In [ ]:
if results:
    # Create analysis directory
    analysis_dir = Path("analysis")
    analysis_dir.mkdir(exist_ok=True)
    
    # Export to CSV
    formatter.to_csv(by_model, analysis_dir / "model_results.csv")
    print("Exported to analysis/model_results.csv")
    
    # Export to JSON
    formatter.to_json(by_model, analysis_dir / "model_results.json")
    print("Exported to analysis/model_results.json")

## 6. Visualizations

Requires viz dependencies: `pip install -e ".[viz]"`

In [ ]:
if results and GRAPH_AVAILABLE:
    grapher = GraphGenerator()
    plots_dir = Path("analysis/plots")
    plots_dir.mkdir(parents=True, exist_ok=True)
    
    # Model comparison
    grapher.plot_model_comparison(
        by_model,
        metric="mean",
        output_path=plots_dir / "model_comparison_mean.png"
    )
    print("Generated: analysis/plots/model_comparison_mean.png")
    
    # Pass rate comparison
    grapher.plot_model_comparison(
        by_model,
        metric="pass_rate",
        output_path=plots_dir / "model_comparison_pass_rate.png"
    )
    print("Generated: analysis/plots/model_comparison_pass_rate.png")
    
    # Score distribution
    grouped = aggregator.group_by_model(results)
    grapher.plot_score_distribution(
        grouped,
        output_path=plots_dir / "score_distribution.png"
    )
    print("Generated: analysis/plots/score_distribution.png")
    
    # Timeline
    grapher.plot_timeline(
        results,
        output_path=plots_dir / "timeline.png"
    )
    print("Generated: analysis/plots/timeline.png")
    
    # Execution time comparison
    grapher.plot_execution_time_comparison(
        by_model,
        output_path=plots_dir / "execution_times.png"
    )
    print("Generated: analysis/plots/execution_times.png")
elif results:
    print("Visualization dependencies not installed. Install with: pip install -e '.[viz]'")

## 7. Per-Task Analysis

In [ ]:
if results:
    # Aggregate by task
    by_task = aggregator.aggregate_task_results(results)
    
    print("\nPer-Task Statistics (showing top 10 by difficulty):\n")
    
    # Sort by mean score (ascending = hardest first)
    sorted_tasks = sorted(by_task.items(), key=lambda x: x[1]['mean'])
    
    for task_id, metrics in sorted_tasks[:10]:
        print(f"{task_id}:")
        print(f"  Mean Score: {metrics['mean']:.3f}")
        print(f"  Pass Rate: {metrics['pass_rate']:.1%}")
        print(f"  Models Tested: {metrics['models_tested']}")
        print()

## 8. Compare Runs (Optional)

Compare a baseline run with a current run to see improvements/regressions.

In [ ]:
# Uncomment and modify with actual file paths
# baseline_files = ["results/baseline_run.json"]
# current_files = ["results/current_run.json"]

# baseline_results = aggregator.aggregate_files(baseline_files)
# current_results = aggregator.aggregate_files(current_files)

# comparison = aggregator.compare_runs(baseline_results, current_results)
# comparison_text = formatter.format_comparison(comparison)
# print(comparison_text)